<a href="https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector
>> all mode; inputs together

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector contains content, search, and engagement signals that are available before the prediction target is evaluated.

I excluded all target-derived variables and identifiers. The remaining feature set includes:

- search_volume
- competition
- cpc
- content_type
- main_intent
- word_count
- char_count
- impressions_90d
- clicks_90d
- ctr
- avg_position
- content_age_days
- engagement_rate
- scroll_rate
- ai_traffic_pct

Numerical missing values will be filled with median values. Categorical variables will be converted using one-hot encoding. This creates a machine-learning-ready feature matrix while preserving the original information.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

github_csv_url = "https://raw.githubusercontent.com/ritakimani9-lang/machinelearning/refs/heads/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(github_csv_url)

# Build target
df["is_declining_label"] = df["trend_pct"] < -20

# Features retained
selected_features = [
    "search_volume",
    "competition",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[selected_features].copy()

# Fill numeric missing values
num_cols = X.select_dtypes(include=["number"]).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

# One-hot encode categoricals
X = pd.get_dummies(X, drop_first=True)

print("Feature matrix shape:", X.shape)
X.head()

Feature matrix shape: (30000, 18)


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,engagement_rate,scroll_rate,ai_traffic_pct,content_type_feedly article,content_type_keyword article,main_intent_informational,main_intent_navigational,main_intent_transactional
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,0.76,10.6,187,5.88,4.55,0.0,False,True,False,False,True
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,0.05,20.3,445,0.00,10.00,0.0,False,True,True,False,False
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,0.09,36.5,141,0.00,28.57,0.0,False,True,True,False,False
3,10.0,0.00,0.00,2877.0,19116.0,11751,58,0.49,6.2,463,1.28,3.45,0.0,False,True,False,False,False
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,0.13,44.0,263,0.00,24.29,0.0,False,True,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Feature Notes

search_volume
- Meaning: Estimated search demand for the keyword.
- Missing values: Filled with median.
- Available before prediction: Yes.

competition
- Meaning: Competitive difficulty score.
- Missing values: Filled with median.
- Available before prediction: Yes.

content_type
- Meaning: Category of content.
- Missing values: Assigned category if needed.
- Available before prediction: Yes.

word_count
- Meaning: Number of words in the content.
- Missing values: Filled with median.
- Available before prediction: Yes.

ctr
- Meaning: Click-through rate from search.
- Missing values: Filled with median.
- Available before prediction: Yes.

avg_position
- Meaning: Average ranking position in search results.
- Missing values: Filled with median.
- Available before prediction: Yes.

engagement_rate
- Meaning: User engagement measure from GA4.
- Missing values: May be unavailable when GA4 tracking is absent.
- Available before prediction: Yes, when collected.

scroll_rate
- Meaning: User scrolling behaviour.
- Missing values: Filled with median.
- Available before prediction: Yes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Potential leakage sources were reviewed systematically.

trend_direction
- Leakage Risk: HIGH
- Reason: Directly used to define the target.

trend_pct
- Leakage Risk: HIGH
- Reason: Target is derived from this value.

impressions_last_30d
- Leakage Risk: HIGH
- Reason: Used in the calculation of trend_pct.

impressions_prev_30d
- Leakage Risk: HIGH
- Reason: Used in the calculation of trend_pct.

content_id
- Leakage Risk: LOW
- Reason: Not target-derived, but acts as an identifier and may cause memorisation.

client_id
- Leakage Risk: LOW
- Reason: Identifier that may allow client-specific memorisation.

No retained feature directly defines or contains the target label. Remaining features represent information that exists independently of the decline classification.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
candidate_leakage_features = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
    "content_id",
    "client_id"
]

print("Potential leakage review:")
for col in candidate_leakage_features:
    print("-", col)


Potential leakage review:
- trend_direction
- trend_pct
- impressions_last_30d
- impressions_prev_30d
- content_id
- client_id


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded Features

trend_direction
- Excluded because it directly defines the target label.

trend_pct
- Excluded because the target is derived from this variable.

impressions_last_30d
- Excluded because it contributes directly to trend_pct and creates leakage risk.

impressions_prev_30d
- Excluded because it contributes directly to trend_pct and creates leakage risk.

content_id
- Excluded because it is an identifier rather than a behavioural signal.

client_id
- Excluded because it is an identifier and may encourage client-specific memorisation instead of general patterns.

provider_used
- Excluded because it reflects operational information rather than content performance.

model_used
- Excluded because it describes the generating system rather than the content itself.

The final feature set contains only variables that could realistically be available before observing the decline outcome.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.